# Наочний приклад: Абстракція, Успадкування, Поліморфізм

Ці три «кити» ООП найкраще зрозуміти на **одному наскрізному прикладі** —
ієрархії **геометричних фігур**. Ми крок за кроком:

1. **Абстракція** — створимо абстрактний клас `Figure`, який описує, *що* вміє будь-яка фігура, але не *як*.
2. **Успадкування** — зробимо конкретні фігури (`Circle`, `Rectangle`, `Square`), що повторно використовують спільний код.
3. **Поліморфізм** — напишемо код, який однаково працює з **будь-якою** фігурою.

> 💡 Запускайте клітинки по черзі й дивіться на результат. Експериментуйте — змінюйте числа, додавайте свої фігури!

---
## 1. Абстракція 🎭

**Абстракція** — це виділення *суттєвого* та приховування зайвих деталей.
Ми описуємо **загальне поняття** «фігура»: будь-яка фігура має **площу** та **периметр**,
але формула в кожної своя.

В Python для цього є модуль `abc` (*Abstract Base Classes*):

- клас успадковує `ABC`;
- метод, позначений декоратором `@abstractmethod`, **обов'язково** має бути реалізований у нащадках;
- сам абстрактний клас **не можна створити** — він лише «контракт»/шаблон.

In [1]:
from abc import ABC, abstractmethod


class Figure(ABC):
    """Абстрактна фігура: описує інтерфейс, але не реалізацію."""

    @abstractmethod
    def area(self) -> float:
        """Площа фігури."""
        ...

    @abstractmethod
    def perimeter(self) -> float:
        """Периметр фігури."""
        ...

    # А цей метод НЕ абстрактний — він спільний для всіх фігур
    def describe(self) -> str:
        return f"{type(self).__name__}: площа = {self.area():.2f}, периметр = {self.perimeter():.2f}"


Спробуємо створити «просто фігуру» — Python не дозволить, бо це лише абстракція:

In [2]:
try:
    f = Figure()          # помилка! абстрактний клас не можна інстанціювати
except TypeError as e:
    print("Помилка:", e)


Помилка: Can't instantiate abstract class Figure without an implementation for abstract methods 'area', 'perimeter'


---
## 2. Успадкування 🧬

**Успадкування** дозволяє нащадку *взяти все* від батьківського класу й **доповнити або змінити** його.

- `class Circle(Figure)` означає «`Circle` — це `Figure`» (відношення **is-a**);
- нащадок зобов'язаний реалізувати всі абстрактні методи (`area`, `perimeter`);
- метод `describe()` нащадки отримують **безкоштовно** — його не треба переписувати.

In [3]:
import math


class Circle(Figure):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return math.pi * self.radius ** 2

    def perimeter(self):
        return 2 * math.pi * self.radius

class Rectangle(Figure):
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height

    def perimeter(self):
        return 2 * (self.width + self.height)


c = Circle(5)
r = Rectangle(4, 6)

print(c.describe())   # метод describe() успадкований від Figure!
print(r.describe())


Circle: площа = 78.54, периметр = 31.42
Rectangle: площа = 24.00, периметр = 20.00


### `super()` — звертання до батьківського класу

`Square` (квадрат) — це окремий випадок прямокутника, тож логічно успадкувати його **від `Rectangle`**,
а не від `Figure`. У конструкторі викликаємо `super().__init__(...)`, щоб **повторно використати**
ініціалізацію батька замість копіювання коду.

In [4]:
class Square(Rectangle):
    def __init__(self, side):
        # квадрат — прямокутник з рівними сторонами
        super().__init__(side, side)   # викликаємо __init__ батька (Rectangle)


sq = Square(3)
print(sq.describe())
print("Площа:", sq.area())          # area() успадковано від Rectangle
print("Це Rectangle?", isinstance(sq, Rectangle))
print("Це Figure?", isinstance(sq, Figure))
print("Square — підклас Figure?", issubclass(Square, Figure))


Square: площа = 9.00, периметр = 12.00
Площа: 9
Це Rectangle? True
Це Figure? True
Square — підклас Figure? True


---
## 3. Поліморфізм 🔀

**Поліморфізм** (грец. «багато форм») — це здатність **одного й того ж коду** працювати
з об'єктами **різних класів**, якщо вони мають спільний інтерфейс.

Нижче функція `total_area` нічого не знає про конкретні фігури — вона просто викликає `area()`.
Кожна фігура **сама вирішує**, як її порахувати. Це і є поліморфізм.

In [7]:
figures = [Square(4)]


def total_area(figs):
    return sum(f.area() for f in figs)   # один виклик area() — різна поведінка


for f in figures:
    print(f.describe())

print("-" * 40)
print(f"Сумарна площа: {total_area(figures):.2f}")


Square: площа = 16.00, периметр = 16.00
----------------------------------------
Сумарна площа: 16.00


### Перевизначення (override) методів і `__str__`

Нащадок може **перевизначити** метод батька — дати йому іншу поведінку.
Часто перевизначають «магічний» метод `__str__`, щоб об'єкт гарно виводився через `print()`.

In [8]:
class LabeledCircle(Circle):
    def __init__(self, radius, label):
        super().__init__(radius)       # беремо radius від Circle
        self.label = label

    def describe(self):                  # перевизначаємо вивід
        return f"АААА! Коло '{self.label}' (r={self.radius})"


lc = LabeledCircle(10, "Сонце")
ci = Circle(4)
print(ci.describe())
print(lc.describe())     # describe() усе ще працює — успадковано


Circle: площа = 50.27, периметр = 25.13
АААА! Коло 'Сонце' (r=10)


### Качина типізація (duck typing) 🦆

Python зазвичай **не вимагає** спільного батька — достатньо, щоб у об'єкта *був потрібний метод*.

> «Якщо щось ходить як качка і крякає як качка — це качка.»

Функція нижче працює з будь-яким об'єктом, у якого є метод `area()` — навіть якщо він
**не** успадкований від `Figure`.

In [ ]:
class Triangle:                     # НЕ успадковує Figure!
    def __init__(self, base, height):
        self.base = base
        self.height = height

    def area(self):
        return 0.5 * self.base * self.height


def print_area(obj):
    # нам байдуже, якого класу obj — головне, щоб був метод area()
    print(f"Площа = {obj.area():.2f}")


print_area(Circle(3))      # Figure
print_area(Triangle(6, 4)) # не Figure, але має area() — і це працює!


---
## Підсумок 📌

| Принцип | Що це | У прикладі |
|---|---|---|
| **Абстракція** | приховати деталі, лишити суть; абстрактний клас = «контракт» | `Figure(ABC)` з `@abstractmethod` |
| **Успадкування** | нащадок бере код батька й доповнює його | `Square(Rectangle)`, `super().__init__()` |
| **Поліморфізм** | один код — різні об'єкти зі спільним інтерфейсом | `total_area()`, перевизначення `__str__`, duck typing |

➡️ Тепер перевірте себе у файлі **`quiz_oop_inheritance_polymorphism.ipynb`**!